<a href="https://colab.research.google.com/github/patilatharvydseit-coder/DSPY-Pratctical-Notebooks/blob/main/MiniProject%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

# ============================================================
# NOTEBOOK 1: Data Selection, Cleaning & Wrangling
# Dataset: Concrete Compressive Strength (UCI ML Repository)
# ============================================================

# ── Imports ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [6]:
df = pd.read_csv('Concrete_Data.csv')

In [7]:
# ── 2. UNDERSTANDING THE ATTRIBUTES ──────────────────────────
# Display raw shape and column names
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())

Shape: (1030, 9)

Columns:
 ['Cement (component 1)(kg in a m^3 mixture)', 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)', 'Fly Ash (component 3)(kg in a m^3 mixture)', 'Water  (component 4)(kg in a m^3 mixture)', 'Superplasticizer (component 5)(kg in a m^3 mixture)', 'Coarse Aggregate  (component 6)(kg in a m^3 mixture)', 'Fine Aggregate (component 7)(kg in a m^3 mixture)', 'Age (day)', 'Concrete compressive strength(MPa, megapascals) ']


In [8]:

# ATTRIBUTE DESCRIPTIONS
# ┌─────────────────────────────────────────────────────────────────────┐
# │ Column Name                              │ Type    │ Unit            │
# ├──────────────────────────────────────────┼─────────┼─────────────────┤
# │ Cement (component 1)                     │ float64 │ kg/m³           │
# │ Blast Furnace Slag (component 2)         │ float64 │ kg/m³           │
# │ Fly Ash (component 3)                    │ float64 │ kg/m³           │
# │ Water  (component 4)                     │ float64 │ kg/m³           │
# │ Superplasticizer (component 5)           │ float64 │ kg/m³           │
# │ Coarse Aggregate  (component 6)          │ float64 │ kg/m³           │
# │ Fine Aggregate (component 7)             │ float64 │ kg/m³           │
# │ Age (day)                                │ int64   │ days (1–365)    │
# │ Concrete compressive strength (MPa) ← TARGET │ float64 │ MPa        │
# └─────────────────────────────────────────────────────────────────────┘
print("\nData Types:\n", df.dtypes)
print("\nFirst 5 Rows:\n", df.head())


Data Types:
 Cement (component 1)(kg in a m^3 mixture)                float64
Blast Furnace Slag (component 2)(kg in a m^3 mixture)    float64
Fly Ash (component 3)(kg in a m^3 mixture)               float64
Water  (component 4)(kg in a m^3 mixture)                float64
Superplasticizer (component 5)(kg in a m^3 mixture)      float64
Coarse Aggregate  (component 6)(kg in a m^3 mixture)     float64
Fine Aggregate (component 7)(kg in a m^3 mixture)        float64
Age (day)                                                  int64
Concrete compressive strength(MPa, megapascals)          float64
dtype: object

First 5 Rows:
    Cement (component 1)(kg in a m^3 mixture)  \
0                                      540.0   
1                                      540.0   
2                                      332.5   
3                                      332.5   
4                                      198.6   

   Blast Furnace Slag (component 2)(kg in a m^3 mixture)  \
0                     

In [9]:
# ── 3. DATA CLEANING ─────────────────────────────────────────

# 3a. Check for missing values
print("\nMissing Values:\n", df.isnull().sum())
# Observation: No missing values — dataset is complete.

# 3b. Check for duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())
duplicates = df[df.duplicated()]
print(duplicates)
# Observation: 25 duplicate rows found → drop them.
df = df.drop_duplicates()
print("\nShape after removing duplicates:", df.shape)

# 3c. Confirm no categorical columns
# All columns are numeric — no encoding needed.
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("\nCategorical Columns:", cat_cols if cat_cols else "None")


Missing Values:
 Cement (component 1)(kg in a m^3 mixture)                0
Blast Furnace Slag (component 2)(kg in a m^3 mixture)    0
Fly Ash (component 3)(kg in a m^3 mixture)               0
Water  (component 4)(kg in a m^3 mixture)                0
Superplasticizer (component 5)(kg in a m^3 mixture)      0
Coarse Aggregate  (component 6)(kg in a m^3 mixture)     0
Fine Aggregate (component 7)(kg in a m^3 mixture)        0
Age (day)                                                0
Concrete compressive strength(MPa, megapascals)          0
dtype: int64

Duplicate Rows: 25
     Cement (component 1)(kg in a m^3 mixture)  \
77                                       425.0   
80                                       425.0   
86                                       362.6   
88                                       362.6   
91                                       362.6   
100                                      425.0   
103                                      425.0   
109               

In [10]:
# ── 4. ATTRIBUTE FILTERING (DIMENSION SELECTION) ─────────────
# WHY KEEP ALL 8 INPUT FEATURES?
# ● Cement           – Primary binder; strong positive effect on strength.
# ● Blast Furnace Slag – Supplementary cementitious material; improves durability.
# ● Fly Ash          – Industrial by-product; reduces water demand.
# ● Water            – Water-to-cement ratio is the single biggest predictor of strength.
# ● Superplasticizer – Admixture that improves workability without adding water.
# ● Coarse Aggregate – Provides bulk volume; affects density.
# ● Fine Aggregate   – Fills voids between coarse particles.
# ● Age              – Strength increases non-linearly with curing time.
#
# No feature is dropped at this stage; all have domain-level justification.
# Feature importance will be evaluated in Notebook 3 (ML stage).


In [11]:
# ── 5. DATA WRANGLING ────────────────────────────────────────

# 5a. Rename columns for readability
df.columns = [
    'Cement', 'BF_Slag', 'Fly_Ash', 'Water',
    'Superplasticizer', 'Coarse_Agg', 'Fine_Agg',
    'Age', 'Strength'
]
print("\nRenamed Columns:", df.columns.tolist())

# 5b. Engineer feature: Cement-to-Water Ratio
# WHY? The w/c ratio is the most important factor in concrete strength
# (lower ratio → higher strength). Creating it explicitly helps the model.
df['Cement_to_Water'] = df['Cement'] / df['Water']

# 5c. Engineer feature: Composite Ratio
# WHY? Captures interaction between supplementary materials and aggregates.
df['Composite'] = df['BF_Slag'] / (df['Fly_Ash'] + df['Coarse_Agg'])

# 5d. Reset index after cleaning
df = df.reset_index(drop=True)


Renamed Columns: ['Cement', 'BF_Slag', 'Fly_Ash', 'Water', 'Superplasticizer', 'Coarse_Agg', 'Fine_Agg', 'Age', 'Strength']


In [12]:
# ── 6. FINAL DATASET SUMMARY ─────────────────────────────────
print("\n── Final Dataset Info ──")
df.info()
print("\nDescriptive Statistics:\n", df.describe().T)
print("\nFinal Shape:", df.shape)

# ── 7. SAVE CLEANED DATA FOR NOTEBOOK 2 ──────────────────────
df.to_csv('cleaned_concrete_data.csv', index=False)
print("\nCleaned dataset saved as 'cleaned_concrete_data.csv'")



── Final Dataset Info ──
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Cement            1005 non-null   float64
 1   BF_Slag           1005 non-null   float64
 2   Fly_Ash           1005 non-null   float64
 3   Water             1005 non-null   float64
 4   Superplasticizer  1005 non-null   float64
 5   Coarse_Agg        1005 non-null   float64
 6   Fine_Agg          1005 non-null   float64
 7   Age               1005 non-null   int64  
 8   Strength          1005 non-null   float64
 9   Cement_to_Water   1005 non-null   float64
 10  Composite         1005 non-null   float64
dtypes: float64(10), int64(1)
memory usage: 86.5 KB

Descriptive Statistics:
                    count        mean         std        min         25%  \
Cement            1005.0  278.631343  104.344261  102.00000  190.700000   
BF_Slag           1005.0   7